
---
# Curso: Data Mining - Semana 2

Docente: Soledad Espezúa (s.espezual@up.edu.pe)

---

<a name='manejoarchivos'></a>
# <font color=blue> Limpieza inicial de datos e integración básica</font>

### <font color=green> 1. Importar datos </font>
Archivos usados:
- `clientes.csv`: datos administrativos de clientes.
- `compras.csv`: datos de compras y comportamiento.

La llave de integración será `id_cliente`.


In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

In [ ]:
clientes = pd.read_csv("clientes.csv")
compras = pd.read_csv("compras.csv")

display(clientes.head())
display(compras.head())

### <font color=green> 2. Inspección inicial </font>

Revisamos tamaño, tipos de datos y estadísticas descriptivas.

In [ ]:
print("Tamaño de clientes:", clientes.shape)
print("Tamaño de compras:", compras.shape)

print("\nTipos detectados en clientes:")
display(clientes.dtypes)

print("\nTipos detectados en compras:")
display(compras.dtypes)

In [ ]:
print("Resumen de clientes:")
display(clientes.describe(include="all").T)

print("Resumen de compras:")
display(compras.describe(include="all").T)


### <font color=green> 3. Valores faltantes </font>

Primero detectamos; todavía no corregimos.

In [ ]:
faltantes_clientes = pd.DataFrame({
    "n_faltantes": clientes.isna().sum(),
    "porcentaje": (clientes.isna().mean() * 100).round(2)
})

faltantes_compras = pd.DataFrame({
    "n_faltantes": compras.isna().sum(),
    "porcentaje": (compras.isna().mean() * 100).round(2)
})

print("Faltantes en clientes")
display(faltantes_clientes)

print("Faltantes en compras")
display(faltantes_compras)


### <font color=green> 4. Categorías y errores de registro </font>

Revisamos valores distintos en variables categóricas. Esto ayuda a encontrar codificaciones no uniformes.

In [ ]:
print("Valores de sexo:")
display(clientes["sexo"].value_counts(dropna=False))

print("Valores de estado_civil:")
display(clientes["estado_civil"].value_counts(dropna=False))

print("Valores de canal:")
display(compras["canal"].value_counts(dropna=False))

print("Valores de cliente_activo:")
display(compras["cliente_activo"].value_counts(dropna=False))

### <font color=green>  5. Duplicados</font>

Distinguimos duplicados exactos y duplicados por clave (`id_cliente`).

In [ ]:
print("Filas duplicadas exactas en clientes:", clientes.duplicated().sum())
print("Filas duplicadas exactas en compras:", compras.duplicated().sum())

print("\nIDs repetidos en clientes:")
display(clientes[clientes.duplicated(subset=["id_cliente"], keep=False)].sort_values("id_cliente"))

print("\nIDs repetidos en compras:")
display(compras[compras.duplicated(subset=["id_cliente"], keep=False)].sort_values("id_cliente"))


### <font color=green>  6. Conversión de tipos</font>

Convertimos variables que deberían ser numéricas o fechas.

`errors="coerce"` transforma valores inválidos en `NaN`, para poder detectarlos y tratarlos.

In [ ]:
clientes["edad"] = pd.to_numeric(clientes["edad"], errors="coerce")
clientes["fecha_registro"] = pd.to_datetime(clientes["fecha_registro"], errors="coerce", dayfirst=True)

compras["monto"] = pd.to_numeric(compras["monto"], errors="coerce")
compras["fecha_compra"] = pd.to_datetime(compras["fecha_compra"], errors="coerce", dayfirst=True)

print("Tipos después de convertir:")
display(clientes.dtypes)
display(compras.dtypes)

### <font color=green> 7. Estandarización de nombres y categorías</font>

Unificamos categorías escritas de distintas formas.

In [ ]:
# Estandarizar texto: quitar espacios y pasar a minúsculas
clientes["sexo"] = clientes["sexo"].astype("string").str.strip().str.lower()
clientes["distrito"] = clientes["distrito"].astype("string").str.strip().str.title()
clientes["estado_civil"] = clientes["estado_civil"].astype("string").str.strip().str.lower()

compras["canal"] = compras["canal"].astype("string").str.strip().str.lower()
compras["cliente_activo"] = compras["cliente_activo"].astype("string").str.strip().str.lower()

# Mapear categorías
clientes["sexo"] = clientes["sexo"].replace({
    "f": "Femenino",
    "femenino": "Femenino",
    "m": "Masculino",
    "masculino": "Masculino",
    "h": "Masculino"
})

clientes["estado_civil"] = clientes["estado_civil"].replace({
    "s": "Soltero",
    "soltero": "Soltero",
    "c": "Casado",
    "casado": "Casado",
    "viudo": "Viudo",
    "separado": "Separado",
    "": pd.NA   #pd.NA es el marcador de faltante "universal" de pandas (funciona igual para texto, enteros o booleanos)
})

compras["canal"] = compras["canal"].replace({
    "online": "Online",
    "web": "Online",
    "tienda": "Tienda"
})

compras["cliente_activo"] = compras["cliente_activo"].replace({
    "sí": "Sí",
    "si": "Sí",
    "s": "Sí",
    "no": "No",
    "n": "No"
})

print("Categorías estandarizadas:")
display(clientes["sexo"].value_counts(dropna=False))
display(clientes["estado_civil"].value_counts(dropna=False))
display(compras["canal"].value_counts(dropna=False))
display(compras["cliente_activo"].value_counts(dropna=False))


### <font color=green> 8. Valores fuera de rango</font>

Detectamos valores imposibles o poco razonables según reglas simples.

In [ ]:
print("Edades fuera de rango:")
display(clientes[(clientes["edad"] < 0) | (clientes["edad"] > 100)])

print("Montos de compra fuera de rango:")
display(compras[(compras["monto"] < 0) | (compras["monto"] > 50000)])


### <font color=green>  9. Tratamiento básico de valores inválidos y faltantes

Primero convertimos valores inválidos en faltantes. Luego aplicamos tratamientos simples.

In [ ]:
# Convertir valores inválidos a NaN
clientes.loc[(clientes["edad"] < 0) | (clientes["edad"] > 100), "edad"] = np.nan

compras.loc[(compras["monto"] < 0) | (compras["monto"] > 50000), "monto"] = np.nan

# Imputación simple
clientes["edad"] = clientes["edad"].fillna(clientes["edad"].median())
clientes["sexo"] = clientes["sexo"].fillna("No especificado")
clientes["distrito"] = clientes["distrito"].fillna("No especificado")
clientes["estado_civil"] = clientes["estado_civil"].fillna("No especificado")

compras["monto"] = compras["monto"].fillna(compras["monto"].median())
compras["canal"] = compras["canal"].fillna("No especificado")
compras["cliente_activo"] = compras["cliente_activo"].fillna("No especificado")

print("Faltantes después del tratamiento:")
display(clientes.isna().sum())
display(compras.isna().sum())

### <font color=green>  10. Eliminación o consolidación de duplicados</font>

Para este laboratorio, conservamos el primer registro por `id_cliente`.

En un proyecto real, esta decisión debe justificarse según la fuente más confiable, la fecha más reciente o la regla de negocio.

In [ ]:
clientes_limpio = clientes.drop_duplicates(subset=["id_cliente"], keep="first")
compras_limpio = compras.drop_duplicates(subset=["id_cliente"], keep="first")

print("Clientes antes y después:", clientes.shape, clientes_limpio.shape)
print("Compras antes y después:", compras.shape, compras_limpio.shape)

### <font color=green>  11. Integración básica</font>

Unimos las fuentes usando `id_cliente`.

Usamos `how="outer"` para conservar todos los registros y ver cuáles aparecen en una fuente, en la otra o en ambas.

In [ ]:
base_integrada = pd.merge(
    clientes_limpio,
    compras_limpio,
    on="id_cliente",
    how="outer",
    suffixes=("_clientes", "_compras"),
    indicator=True
)

display(base_integrada)
display(base_integrada["_merge"].value_counts())


### <font color=green>  12. Validación final</font>

Después de limpiar e integrar, volvemos a revisar tipos, faltantes y resumen.

In [ ]:
print("Tipos finales:")
display(base_integrada.dtypes)

print("Faltantes finales:")
display(base_integrada.isna().sum())

print("Resumen final:")
display(base_integrada.describe(include="all"))


### <font color=green> 13. Registro de decisiones de limpieza</font>

En un proyecto real, no basta con limpiar: hay que justificar cada decisión.

In [ ]:
decisiones = pd.DataFrame([
    ["sexo", "Categorías no uniformes", "Estandarizar F/M/femenino/masculino/H", "Facilita análisis y evita categorías duplicadas."],
    ["edad", "Valor fuera de rango", "Convertir edad > 100 en faltante e imputar mediana", "160 años no es razonable para este contexto."],
    ["ingreso_mensual", "Valores negativos o extremos", "Convertir inválidos en faltantes e imputar mediana", "Evita distorsionar la media."],
    ["monto_compra", "Valores negativos o extremos", "Convertir inválidos en faltantes e imputar mediana", "Un monto negativo o enorme requiere validación."],
    ["id_cliente", "Duplicados por llave", "Conservar primer registro", "Decisión simplificada para el laboratorio; en proyecto real debe justificarse mejor."]
], columns=["variable", "problema", "decisión", "justificación"])

display(decisiones)


# Desafío para grupos

Aplicar este mismo flujo a las fuentes de datos de su proyecto.

## Entregables mínimos

1. **Descripción del problema y unidad de análisis.**
2. **Fuentes de datos usadas** y llave posible de integración.
3. **Tabla resumen de cada fuente**: tamaño, tipos, faltantes, valores distintos y ejemplos de valores.
4. **Problemas detectados**:
   - valores faltantes;
   - duplicados;
   - errores de registro;
   - tipos incorrectos;
   - valores fuera de rango;
   - posibles outliers;
   - categorías no uniformes.
5. **Decisiones de limpieza justificadas**:
   - qué se corrigió;
   - qué se imputó;
   - qué se eliminó;
   - qué se mantuvo y por qué.
6. **Base integrada inicial** o plan claro de integración.

## Presentación breve

Cada grupo tendrá 3 a 5 minutos para mostrar:
- una captura o tabla de inspección;
- dos problemas encontrados;
- dos decisiones de limpieza justificadas;
- la clave (ID) o estrategia de integración.
